In [ ]:
# =========================================================
# 0) Imports
# =========================================================
import polars as pl              # fast DataFrame engine
import numpy as np
from pathlib import Path
import lightgbm as lgb
from   scipy.stats import rankdata   # for row-wise ranking

SOLUTION_NULL_FILLER = -999999
ROW_ID_COL           = "date_id"      # change if the column is named differently



# =========================================================
# 2) Read data with Polars
# =========================================================
DATA_DIR = Path("/kaggle/input/mitsui-commodity-prediction-challenge")

# ---------- load ----------
feat_pl   = pl.read_csv(DATA_DIR / "train.csv")          # features   (has date_id)
label_pl  = pl.read_csv(DATA_DIR / "train_labels.csv")   # targets    (has date_id)
test_pl   = pl.read_csv(DATA_DIR / "test.csv")           # test feats (has date_id)

# ---------- join on `date_id` ----------
train_pl = feat_pl.join(label_pl, on="date_id", how="inner")

# ---------- column sets ----------
target_cols  = [c for c in train_pl.columns if c.startswith("target_")]
feature_cols = [c for c in train_pl.columns
                if c not in target_cols + ["date_id"]]


# fill all feature nulls with 0.0  (or median/mean per column)
train_pl = train_pl.with_columns(
    [pl.col(c).fill_null(0.0) for c in feature_cols]
)
test_pl  = test_pl.with_columns(
    [pl.col(c).fill_null(0.0) for c in feature_cols]
)

# ---------- numpy matrices for LightGBM ----------
X_train = train_pl.select(feature_cols).to_numpy()
y_dict  = {t: train_pl.select(t).to_numpy().ravel().astype(float) 
           for t in target_cols}

X_test  = test_pl.select(feature_cols).to_numpy()

# check the data
print("train_pl shape:", train_pl.shape)      # rows, cols
print("X_train shape:", X_train.shape, X_train.dtype)  # (n_rows, n_features)
print("Example target 0 shape:", y_dict[target_cols[0]].shape)



assert not np.isnan(X_train).any(),  "NaNs in features!"
assert X_train.dtype in (np.float32, np.float64), "LightGBM wants float"


# ---------- LightGBM params ----------
params = dict(
    objective        = "regression",
    boosting_type    = "gbdt",
    n_estimators     = 4000,
    learning_rate    = 0.02,
    num_leaves       = 127,
    feature_fraction = 0.8,
    subsample        = 0.7,
    lambda_l2        = 2e-3,
    random_state     = 42,
    verbose          = -1,
    n_jobs           = 2,            # 2 vCPU on Kaggle
)

#params.update({
#    "device_type":     "gpu",
#    "max_bin":         255,
#    "gpu_platform_id": 0,
#    "gpu_device_id":   0,
#})

# ---------- train one model per target ----------
#  the built-in callback (LightGBM ≥4.0)
stop_cb = lgb.early_stopping(
    stopping_rounds=200,
    first_metric_only=False,   # or True if you monitor the first metric only
    verbose=False,
)

print("debugging")
models = {}
split = int(0.8 * X_train.shape[0])         # 80 % → train, 20 % → val
train_idx = slice(0, split)                 # rows 0 … split-1
val_idx   = slice(split, None)              # rows split … end


for k, tgt in enumerate(target_cols, 1):
    dtrain = lgb.Dataset(X_train[train_idx], label=y_dict[tgt][train_idx])
    dval   = lgb.Dataset(X_train[val_idx],   label=y_dict[tgt][val_idx])
    models[tgt] = lgb.train(
        params,
        dtrain,
        valid_sets=[dval],
        callbacks=[stop_cb],
    )
    if k % 25 == 0:          # progress print every 25 models
        print(f"{k}/{len(target_cols)}  best_iter={models[tgt].best_iteration:4}")


print("debugging")
# ---------- predict ----------
pred_dict = {"date_id": test_pl["date_id"]}
for tgt in target_cols:
    pred_col                       = tgt.replace("target_", "prediction_")
    pred_dict[pred_col]            = models[tgt].predict(X_test)

submission_pl = pl.DataFrame(pred_dict)
submission_pl.write_csv("lightgbm_polars_submission.csv")


# =========================================================
# 7) Optional: quick local validation using an 80/20 split
# =========================================================
from sklearn.model_selection import train_test_split
idx = np.arange(len(train_pl))
tr_idx, v_idx = train_test_split(idx, test_size=0.2, shuffle=False)

val_pred_dict = {"date_id": train_pl["date_id"].to_numpy()[v_idx]}

for tgt in target_cols:
    pred_name = tgt.replace("target_", "prediction_")
    val_pred_dict[pred_name] = models[tgt].predict(X_train_np[v_idx])

val_pred_pl = pl.DataFrame(val_pred_dict)

val_score = score(
    solution_pl= train_pl.select(["date_id"] + target_cols).slice(v_idx[0], len(v_idx)),
    submission_pl= val_pred_pl,
    row_id_column_name= "date_id"
)
print(f"Local Sharpe score (hold-out): {val_score:.4f}")


: 